# Notebook 01 — Production Structure Pipeline

Notebook này có một pipeline production duy nhất. Mỗi fresh runtime chạy preflight và real smoke một video từ Hugging Face trong namespace test riêng; chỉ khi smoke PASS mới xử lý batch được giao, package và sync Phase01. Chỉ sửa cell `USER SETTINGS`, sau đó **Run All**.

Phase01 v1.4 dùng NVIDIA FastConformer cho ASR, Vintern-1B-v3_5 cho OCR, Qwen2.5-VL-7B-Instruct 4-bit cho semantic và Vintern-3B-R-beta làm local fallback. Smoke gọi chính production core, không có pipeline test thứ hai.

Business logic, dependency/runtime preflight, fixture HF, checkpoint/release test isolation, report và cleanup đều nằm trong package. Notebook chỉ clone/update, cài package và gọi CLI bằng fresh Python subprocess để tránh binary/module state cũ trong kernel.

Repo được clone lần đầu và fast-forward tương đương `git pull --ff-only` ở các lần sau.


In [ ]:
# USER SETTINGS — teammate chỉ sửa các giá trị trong cell này, không sửa các cell pipeline bên dưới.
# Mỗi người nhận đúng batch do Notebook 00B tạo tại <release>/manifests/batch_000.txt, batch_001.txt, ...
batch_id = "batch_000"
worker_id = "worker_000"

# Giữ None để auto-resolve Phase00 release mới nhất; chỉ override khi cả team chủ động pin cùng snapshot.
release_id_override = None

# Production HF storage. None dùng versioned defaults trong configs/storage.yaml.
hf_release_repo = None
hf_checkpoint_repo = None
hf_revision = None
hf_prefix = None
checkpoint_revision = None
checkpoint_prefix = None

# ASR production: None hoặc nemo dùng NVIDIA Parakeet; faster_whisper là override có chủ đích.
asr_provider = None
scratch_dir_override = None

# REAL SMOKE GATE — mặc định mỗi fresh worker/session đều chạy một lần trước full batch.
run_real_smoke = True
# None dùng retention policy trong configs/phase01.yaml.
keep_remote_smoke_artifacts = None
cleanup_local_smoke = None

# SOURCE CODE — canonical production notebook chạy code từ branch dev.
github_repo_url = "https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git"
github_branch = "dev"
repo_dir_name = "Multimodal-Agentic-Retrieval-Engine"

# Secrets không đặt trực tiếp trong notebook. HF_TOKEN hoặc AIC_HF_TOKEN bắt buộc để restore/sync.


## Cách teammate chạy Notebook 01

Chỉ sửa `USER SETTINGS`, sau đó chọn **Run All**. `batch_id` phải khớp manifest do Notebook 00B tạo; `worker_id` nên dùng cùng số batch.

Mỗi fresh Colab/Kaggle/local session mặc định chạy `run_real_smoke = True`. Package lấy fixture `L30_V040` đã pin từ Hugging Face, ghi checkpoint vào `AIOU26_checkpoints_test/_smoke/<run_id>` và output vào `AIOU26_release_test/_smoke/<run_id>`. Smoke phải chạy đủ 10 stage với `source=computed`, validate package và remote checksum; fail sẽ chặn full batch.

`keep_remote_smoke_artifacts = None` và `cleanup_local_smoke = None` dùng policy trong `configs/phase01.yaml`. Shared model cache luôn được giữ để full batch không tải lại model. Có thể đặt `run_real_smoke = False` chỉ khi chính runtime hiện tại đã được verify và người chạy chủ động skip.

Production vẫn dùng `hf_release_repo` và `hf_checkpoint_repo` trong USER SETTINGS (giữ `None` để dùng default). Smoke không ghi vào các namespace production này. `HF_TOKEN` hoặc `AIC_HF_TOKEN` phải có quyền đọc fixture và ghi cả repo test lẫn repo production.

Package được cài bằng `pip install -e system1[phase01-production]` và mọi preflight/smoke/full inference chạy trong fresh subprocess. Notebook kernel không import NumPy, Pandas, Torch, Transformers hoặc NeMo sau khi pip thay dependency.


In [ ]:
# BƯỚC 1: Detect environment, paths và secrets. Không in secret.
import os, shutil, sys
from pathlib import Path

if "google.colab" in sys.modules:
    runtime_env = "colab"
    workspace = Path("/content/aic_phase01")
elif "KAGGLE_URL_BASE" in os.environ or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    runtime_env = "kaggle"
    workspace = Path("/kaggle/temp/aic_phase01")
else:
    runtime_env = "local"
    workspace = Path.cwd() / ".phase01_runtime"
workspace.mkdir(parents=True, exist_ok=True)
output_root = workspace / "output"
scratch_dir = Path(scratch_dir_override).expanduser() if scratch_dir_override else workspace / "scratch"
model_cache = workspace / "model_cache"
for path in (output_root, scratch_dir, model_cache): path.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(workspace).free / (1024**3)
if runtime_env in {"colab", "kaggle"} and free_gb < 35:
    raise RuntimeError(f"Không đủ runtime disk cho model local mặc định: {free_gb:.1f} GiB trống < 35 GiB.")

def load_secret(name):
    if os.environ.get(name): return os.environ[name]
    if runtime_env == "colab":
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    if runtime_env == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        except Exception:
            return None
    return None

for secret_name in ("HF_TOKEN", "AIC_HF_TOKEN"):
    value = load_secret(secret_name)
    if value: os.environ[secret_name] = value
if not os.environ.get("HF_TOKEN") and os.environ.get("AIC_HF_TOKEN"): os.environ["HF_TOKEN"] = os.environ["AIC_HF_TOKEN"]
if os.environ.get("HF_TOKEN"): os.environ["AIC_HF_TOKEN"] = os.environ["HF_TOKEN"]
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("Cần cấu hình HF_TOKEN trong Colab/Kaggle Secrets hoặc environment.")

os.environ["HF_HOME"] = str(model_cache / "hf")
os.environ.setdefault("HF_XET_CHUNK_CACHE_SIZE_BYTES", "0")
os.environ.setdefault("HF_XET_SHARD_CACHE_SIZE_LIMIT", str(1024**3))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["AIC_DATA_ROOT"] = str(workspace / "data")
os.environ["AIC_RUNTIME_ROOT"] = str(workspace / "runtime")
os.environ["AIC_ARTIFACT_ROOT"] = str(workspace / "artifacts")
print({"environment": runtime_env, "workspace": str(workspace), "output_root": str(output_root), "scratch": str(scratch_dir)})

In [ ]:
# BƯỚC 2: Clone/update đúng branch GitHub mà không xóa thay đổi local.
import subprocess

def run_command(command, cwd=None):
    print("RUN:", " ".join(map(str, command)))
    result = subprocess.run(command, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode: raise RuntimeError(f"Command failed ({result.returncode}): {command}")
    return result

def is_repo_root(path): return (path / "system1" / "src" / "system1" / "__init__.py").is_file()
repo_root = next((path for path in [Path.cwd(), *Path.cwd().parents] if is_repo_root(path)), None)
if repo_root is None:
    repo_root = workspace / repo_dir_name
    if not (repo_root / ".git").exists():
        run_command(["git", "clone", "--branch", github_branch, "--single-branch", github_repo_url, str(repo_root)], cwd=workspace)
if run_command(["git", "status", "--porcelain"], cwd=repo_root).stdout.strip():
    raise RuntimeError("Repo có thay đổi local; notebook không tự reset/stash.")
run_command(["git", "fetch", "origin", github_branch], cwd=repo_root)
local_branch = subprocess.run(["git", "show-ref", "--verify", "--quiet", f"refs/heads/{github_branch}"], cwd=repo_root).returncode == 0
run_command(["git", "switch", github_branch] if local_branch else ["git", "switch", "--track", "-c", github_branch, f"origin/{github_branch}"], cwd=repo_root)
run_command(["git", "merge", "--ff-only", f"origin/{github_branch}"], cwd=repo_root)
git_sha = run_command(["git", "rev-parse", "HEAD"], cwd=repo_root).stdout.strip()
remote_sha = run_command(["git", "rev-parse", f"origin/{github_branch}"], cwd=repo_root).stdout.strip()
actual_branch = run_command(["git", "branch", "--show-current"], cwd=repo_root).stdout.strip()
dirty = bool(run_command(["git", "status", "--porcelain"], cwd=repo_root).stdout.strip())
source_identity = {"git_commit_sha": git_sha, "local_branch": actual_branch, "expected_branch": github_branch, "origin_branch_sha": remote_sha, "dirty": dirty}
print("SOURCE IDENTITY:", source_identity)
if dirty or actual_branch != github_branch or git_sha != remote_sha:
    raise RuntimeError(f"Source code stale/mismatch; notebook will not reset local work: {source_identity}")
os.environ["AIC_EXPECTED_GIT_BRANCH"] = github_branch
os.environ["AIC_REPO_ROOT"] = str(repo_root)
os.environ["AIC_REPO_PARENT"] = str(repo_root.parent)

In [ ]:
# BƯỚC 3: Cài System1 production package và không import heavy dependency trong kernel.
system1_root = repo_root / "system1"
run_command([sys.executable, "-m", "pip", "install", "-q", "-e", f"{system1_root}[phase01-production]"])
print("System1 installed. Runtime checks sẽ chạy trong fresh subprocess ở bước tiếp theo.")


In [ ]:
# BƯỚC 4: Smoke configuration nằm trong package; cell này chỉ hiển thị lựa chọn launcher.
print({
    "run_real_smoke": run_real_smoke,
    "keep_remote_smoke_artifacts": keep_remote_smoke_artifacts,
    "cleanup_local_smoke": cleanup_local_smoke,
    "smoke_source": "HF pinned fixture from configs/phase01.yaml",
})


In [ ]:
# BƯỚC 5: Helper CLI chạy package trong fresh subprocess, có streaming output và bounded error tail.
from collections import deque

def run_cli(arguments):
    command = [sys.executable, "-m", "system1.cli", *map(str, arguments)]
    env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(command, cwd=system1_root, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    tail = deque(maxlen=120)
    for line in process.stdout:
        print(line, end="", flush=True); tail.append(line)
    code = process.wait()
    if code:
        raise RuntimeError(f"CLI failed ({code}). Last output:\n" + "".join(tail))
    return "".join(tail)

In [ ]:
# BƯỚC 6: Một lệnh package — runtime preflight, real smoke, rồi full assigned batch nếu smoke PASS.
command = [
    "phase01-worker-run", "--batch-id", batch_id, "--worker-id", worker_id,
    "--output", str(output_root), "--scratch-dir", str(scratch_dir),
    "--restore-phase00", "--validate-remote", "--sync",
    "--run-real-smoke" if run_real_smoke else "--skip-real-smoke",
]
if keep_remote_smoke_artifacts is not None:
    command.append("--keep-remote-smoke-artifacts" if keep_remote_smoke_artifacts else "--delete-remote-smoke-artifacts")
if cleanup_local_smoke is not None:
    command.append("--cleanup-local-smoke" if cleanup_local_smoke else "--keep-local-smoke")
optional = {
    "--asr-provider": asr_provider,
    "--release-id-override": release_id_override,
    "--hf-release-repo": hf_release_repo,
    "--hf-checkpoint-repo": hf_checkpoint_repo,
    "--hf-release-revision": hf_revision,
    "--hf-release-prefix": hf_prefix,
    "--checkpoint-revision": checkpoint_revision,
    "--checkpoint-prefix": checkpoint_prefix,
}
for option, value in optional.items():
    if value not in (None, ""): command.extend([option, str(value)])
run_cli(command)


In [ ]:
# BƯỚC 7: Báo cáo ngắn sau Run All.
import json
worker_run = json.loads((output_root / "phase01_worker_last_run.json").read_text())
last_run = json.loads((output_root / "phase01_last_run.json").read_text())
release_root = Path(last_run["release_dir"])
resolved = json.loads((release_root / "manifests/phase01/resolved_config.json").read_text())
report_path = release_root / "manifests/worker_reports" / f"structure_{batch_id}_{worker_id}.json"
review_path = release_root / "manifests/phase01" / f"manual_review_{batch_id}_{worker_id}.json"
report = json.loads(report_path.read_text())
review = json.loads(review_path.read_text())
print({
    "smoke": worker_run.get("smoke"),
    "release_id": resolved["runtime"]["release_id"],
    "config_hash": resolved["config_hash"],
    "counts": report.get("counts"),
    "manual_review_status": review["status"],
    "manual_review_samples": review["sample_size_actual"],
    "report": str(report_path),
})


In [ ]:
# BƯỚC 8: Verify trực tiếp output Phase01 đã hiện diện đầy đủ trên Hugging Face.
from huggingface_hub import HfApi

release_storage = resolved["storage"]["release"]
release_id = resolved["runtime"]["release_id"]
remote_root = f"{release_id}/phase01_structure"
prefix = str(release_storage.get("prefix") or "").strip("/")
scoped_root = f"{prefix}/{remote_root}" if prefix else remote_root
api = HfApi(token=os.environ.get("AIC_HF_TOKEN") or os.environ.get("HF_TOKEN"))
entries = api.list_repo_tree(
    repo_id=release_storage["repo_id"],
    repo_type=release_storage.get("repo_type", "dataset"),
    revision=release_storage.get("revision", "main"),
    path_in_repo=scoped_root,
    recursive=True,
)
remote_files = {entry.path for entry in entries if getattr(entry, "path", None)}
complete_videos = [row["video_id"] for row in report.get("videos", []) if str(row.get("status", "")).startswith("complete")]
package_root = resolved["artifact"]["package"]["root"].format(
    release_id=release_id, batch_id=batch_id, video_id=""
).strip("/")
package_filename = resolved["artifact"]["package"]["filename"]
expected = {f"{prefix}/{package_root}/{package_filename.format(video_id=video_id)}" if prefix else f"{package_root}/{package_filename.format(video_id=video_id)}" for video_id in complete_videos}
expected.update({
    f"{scoped_root}/worker_reports/structure_{batch_id}_{worker_id}.json",
    f"{scoped_root}/worker_reports/errors_{batch_id}_{worker_id}.jsonl",
    f"{scoped_root}/worker_reports/manual_review_{batch_id}_{worker_id}.json",
})
missing = sorted(expected - remote_files)
if missing: raise RuntimeError("HF Phase01 output verification failed; missing: " + ", ".join(missing))
print({
    "hf_repo": release_storage["repo_id"],
    "remote_root": scoped_root,
    "verified_files": len(expected),
    "verified_packages": len(complete_videos),
    "status": "HF phase01_structure verification: OK",
})
